# expand_nested_no 函数演示

本 notebook 演示 `expand_nested_no` 函数的使用，它实现了 OPEdefs.m 风格的嵌套正规序展开。

## 与 simplify 的区别

- **`simplify()`**: 完全规范化，将所有嵌套 NO 规约到"标准基底"（通常是二重 NO + 导数项）
- **`expand_nested_no()`**: 只展开嵌套结构，保留中间形式（如三重嵌套 NO），不进行进一步规约

In [ ]:
from pyope import BasisOperator, OPE, NO, simplify, expand_nested_no, One, d
from pyope.api import MakeOPE

## 示例 1: Kac-Moody 代数 sl(2) at level k=1

定义 sl(2) Kac-Moody 代数的生成元和 OPE 关系：

In [ ]:
J_plus = BasisOperator("J⁺", conformal_weight=1)
J_zero = BasisOperator("J⁰", conformal_weight=1)
J_minus = BasisOperator("J⁻", conformal_weight=1)

k_val = 1  # level

# 定义 OPE 关系
OPE[J_plus, J_zero] = MakeOPE([-2 * J_plus])
OPE[J_plus, J_minus] = MakeOPE([k_val * One, J_zero])
OPE[J_zero, J_minus] = MakeOPE([-2 * J_minus])
OPE[J_zero, J_zero] = MakeOPE([2 * k_val * One, 0])

### 计算嵌套正规序的展开

考虑表达式 `NO(J⁻, NO(J⁺, J⁰))`：

In [ ]:
expr = NO(J_minus, NO(J_plus, J_zero))
print("原始表达式:")
expr

### 使用 expand_nested_no（OPEdefs 风格）

In [ ]:
expanded = expand_nested_no(expr)
print("expand_nested_no 的结果:")
expanded

**结果包含三项：**
1. `NO(J⁺, NO(J⁰, J⁻))` - 三重嵌套项（保留结构）
2. `2*NO(J⁺, ∂J⁻)` - 收缩项
3. `-NO(∂J⁰, J⁰)` - 另一个收缩项

这与 Mathematica OPEdefs 的输出一致。

### 使用 simplify（完全规约）

In [ ]:
simplified = simplify(expr)
print("simplify 的结果:")
simplified

**结果只有一项：**
- `2*NO(J⁺, ∂J⁻)` - 完全规约后的结果

`simplify` 将三重嵌套项进一步规约，得到更简洁的形式。

### 比较两种方法

In [ ]:
print("expand_nested_no:", expanded)
print("\nsimplify:", simplified)
print("\n两者是否相同:", str(expanded) == str(simplified))

## 示例 2: 理解展开过程

让我们逐步理解 `expand_nested_no` 的展开过程：

In [ ]:
print("步骤 1: 原始表达式")
print("  NO(J⁻, NO(J⁺, J⁰))")
print()

print("步骤 2: 应用 Thielemans 重排公式")
print("  NO(B, NO(A,C)) = sign * NO(A, NO(B,C)) + NO(NOCommuteHelp[B,A], C)")
print("  其中 B=J⁻, A=J⁺, C=J⁰")
print()

print("步骤 3: 计算 NOCommuteHelp[J⁻, J⁺]")
print("  根据 OPE[J⁻, J⁺]，得到 -∂J⁰")
print()

print("步骤 4: 展开为")
print("  NO(J⁺, NO(J⁻, J⁰)) + NO(-∂J⁰, J⁰)")
print("  = NO(J⁺, NO(J⁻, J⁰)) - NO(∂J⁰, J⁰)")
print()

print("步骤 5: 重排内层 NO(J⁻, J⁰) -> NO(J⁰, J⁻) + 收缩项")
print("  根据 OPE[J⁰, J⁻]，得到 NO(J⁰, J⁻) + 2*∂J⁻")
print()

print("步骤 6: 最终结果")
print("  NO(J⁺, NO(J⁰, J⁻)) + 2*NO(J⁺, ∂J⁻) - NO(∂J⁰, J⁰)")
print()

print("验证:")
print(expanded)

## 示例 3: 深度控制

`expand_nested_no` 支持深度控制参数 `max_depth`：

In [ ]:
A = BasisOperator("A", conformal_weight=1)
B = BasisOperator("B", conformal_weight=1)
C = BasisOperator("C", conformal_weight=1)

# 设置简单的 OPE（无收缩）
OPE[A, B] = MakeOPE([])
OPE[A, C] = MakeOPE([])
OPE[B, C] = MakeOPE([])

expr = NO(NO(A, B), C)

print("原始表达式:", expr)
print()

print("深度 0 (不展开):", expand_nested_no(expr, max_depth=0))
print("深度 1 (展开一层):", expand_nested_no(expr, max_depth=1))
print("深度 None (完全展开):", expand_nested_no(expr, max_depth=None))

## 总结

- `expand_nested_no()` 提供了 OPEdefs.m 风格的展开，保留三重嵌套结构
- `simplify()` 提供了完全规约，得到最简形式
- 两种方法适用于不同的场景：
  - 需要查看中间展开步骤时，使用 `expand_nested_no()`
  - 需要最终简化结果时，使用 `simplify()`